# 🔮 Módulo 4: Simulador de Inferencia de LLM
## Notebook de Conocimiento Guiado

**Objetivo:** Entender cómo funcionan los servidores de inferencia de LLMs
(vLLM, TGI, TensorRT-LLM) sin ejecutar un modelo real.

**Lore:** Eres el arquitecto del Oráculo Galáctico, el sistema que responde
consultas de millones de civilizaciones. Cada token cuesta energía.

| Sección | Concepto |
|---------|---------|
| 📚 Teoría | Inferencia autoregresiva, KV Cache, Batching |
| 🔨 Demo | KVCache, TokenGenerator, BatchScheduler |
| ✏️ Ejercicio 1 | `MetricsCollector` — p50/p95/p99 latency |
| ✏️ Ejercicio 2 | `PriorityScheduler` — cola de prioridad VIP |
| 🎯 Quiz | Preguntas de Anthropic/OpenAI sobre sistemas |

## 📚 Parte 1: Inferencia Autoregresiva

Los LLMs generan texto **un token a la vez**:

```
Prompt: "La gravedad es"
Token 1: "una"      → "La gravedad es una"
Token 2: "fuerza"   → "La gravedad es una fuerza"
Token 3: "que"      → "La gravedad es una fuerza que"
...
```

**El problema de O(n²):** Para generar el token N, el transformer necesita
atender a los N-1 tokens anteriores. Sin optimización, calcular los N tokens
anteriores N veces = O(n²) operaciones.

**La solución — KV Cache:**
```
En cada capa del transformer, guardamos los vectores K (Key) y V (Value)
de los tokens ya procesados. Para el token N, solo calculamos K,V del
token N y los CONCATENAMOS con los anteriores: O(1) por token.

Sin KV Cache: O(n²) total
Con KV Cache: O(n) total
```

In [ ]:
# 🔨 IMPLEMENTACIÓN: SimpleKVCache
from typing import Any, Dict, List, Optional
import uuid

class SimpleKVCache:
    """
    Simula el KV Cache de un transformer.
    En un sistema real, guarda tensores PyTorch de las capas de atención.
    Aquí guardamos listas de enteros como simulación.
    """
    def __init__(self, max_seq_length: int = 2048):
        self.max_seq_length = max_seq_length
        # request_id → {'keys': [...], 'values': [...], 'tokens_generados': int}
        self._cache: Dict[str, Dict[str, Any]] = {}
    
    def create_session(self, request_id: str, prompt_tokens: List[int]):
        """Inicializa la sesión con los tokens del prompt."""
        if len(prompt_tokens) > self.max_seq_length:
            raise ValueError(f"Prompt muy largo: {len(prompt_tokens)} > {self.max_seq_length}")
        self._cache[request_id] = {
            "keys":   list(prompt_tokens),
            "values": list(prompt_tokens),
            "tokens_generados": 0,
        }
    
    def extend(self, request_id: str, new_token: int) -> bool:
        """Agrega un nuevo token generado al caché. Retorna False si está lleno."""
        session = self._cache.get(request_id)
        if session is None: return False
        if len(session["keys"]) >= self.max_seq_length: return False
        session["keys"].append(new_token)
        session["values"].append(new_token)
        session["tokens_generados"] += 1
        return True
    
    def get_context_length(self, request_id: str) -> int:
        """Retorna el número total de tokens en contexto."""
        return len(self._cache.get(request_id, {}).get("keys", []))
    
    def free(self, request_id: str):
        """Libera la memoria del caché para esta petición."""
        self._cache.pop(request_id, None)
    
    @property
    def memory_usage(self) -> int:
        """Número total de slots de caché en uso."""
        return sum(len(s["keys"]) for s in self._cache.values())

# Demo
cache = SimpleKVCache(max_seq_length=10)
req_id = "req-001"
cache.create_session(req_id, [1, 2, 3, 4])   # Tokens del prompt
print(f"Contexto inicial: {cache.get_context_length(req_id)} tokens")

for token in [5, 6, 7]:
    ok = cache.extend(req_id, token)
    print(f"Extender con token {token}: {'✓' if ok else '✗'} | contexto: {cache.get_context_length(req_id)}")

cache.free(req_id)
print(f"Memoria después de free: {cache.memory_usage} tokens")

## 🏗️ Parte 2: Batch Scheduling

El batching agrupa varias peticiones para procesarlas juntas en la GPU,
maximizando el paralelismo hardware.

**Tradeoff:**
- Batch más grande → Mayor throughput (tokens/segundo)
- Batch más grande → Mayor latencia de la primera respuesta
- **Continuous batching** (vLLM): agrega/elimina requests dinámicamente

In [ ]:
# 🔨 IMPLEMENTACIÓN: TokenGenerator + SimpleBatchScheduler
import random
import time
from collections import deque
from dataclasses import dataclass, field

@dataclass
class Request:
    id: str
    prompt: str
    max_tokens: int = 50
    tokens_generados: int = 0
    resultado: list = field(default_factory=list)
    
    @property
    def completado(self):
        return self.tokens_generados >= self.max_tokens

class TokenGenerator:
    """Simula la generación de tokens (sin modelo real)."""
    VOCABULARIO = ["el", "la", "un", "una", "sistema", "funciona", "bien",
                   "con", "datos", "del", "modelo", "genera", "tokens", "nuevos"]
    
    def generate_next_token(self, request: Request) -> str:
        """Genera el siguiente token. Simula latencia de GPU."""
        time.sleep(0.001)   # ~1ms por token (muy optimista para demo)
        return random.choice(self.VOCABULARIO)

class SimpleBatchScheduler:
    """Agrupa peticiones en batches para procesamiento eficiente."""
    def __init__(self, max_batch_size=4):
        self.max_batch_size = max_batch_size
        self.cola: deque = deque()
        self.activos: dict = {}
    
    def submit(self, request: Request):
        self.cola.append(request)
    
    def step(self, generator: TokenGenerator) -> list:
        """Ejecuta un paso de generación para el batch actual."""
        # Rellenar el batch con requests de la cola
        while len(self.activos) < self.max_batch_size and self.cola:
            req = self.cola.popleft()
            self.activos[req.id] = req
        
        # Generar un token por cada request activo
        completados = []
        for req_id, req in list(self.activos.items()):
            token = generator.generate_next_token(req)
            req.resultado.append(token)
            req.tokens_generados += 1
            if req.completado:
                completados.append(self.activos.pop(req_id))
        
        return completados

# Demo: 3 peticiones con batch size 2
gen = TokenGenerator()
scheduler = SimpleBatchScheduler(max_batch_size=2)

reqs = [Request(f"req-{i}", f"Explica el concepto {i}", max_tokens=5) for i in range(3)]
for r in reqs: scheduler.submit(r)

todos_completados = []
pasos = 0
while scheduler.activos or scheduler.cola:
    completados = scheduler.step(gen)
    todos_completados.extend(completados)
    pasos += 1

print(f"Completados en {pasos} pasos:")
for req in todos_completados:
    print(f"  {req.id}: {' '.join(req.resultado)}")

## ✏️ Ejercicio 1: `MetricsCollector`

**Problema:** Implementa un recolector de métricas que calcule:
- Tokens por segundo (throughput)
- Percentiles de latencia: p50, p95, p99

In [ ]:
# ✏️ EJERCICIO 1 — Completa el TODO
import time

class MetricsCollector:
    """Recopila métricas de rendimiento del servidor de inferencia."""
    
    def __init__(self):
        self.latencias = []       # duración de cada request (segundos)
        self.tokens_totales = 0
        self.inicio = time.time()
    
    def registrar(self, duracion_segundos: float, tokens_generados: int):
        """Registra una request completada."""
        self.latencias.append(duracion_segundos)
        self.tokens_totales += tokens_generados
    
    def tokens_por_segundo(self) -> float:
        # TODO: Retorna el throughput total
        pass
    
    def percentil(self, p: float) -> float:
        """Retorna el percentil p (0-100) de latencias."""
        # TODO: Implementar sin numpy
        # Pista: ordena la lista y toma el índice correcto
        pass
    
    def resumen(self):
        print(f"Throughput: {self.tokens_por_segundo():.1f} tokens/s")
        print(f"Latencia p50: {self.percentil(50)*1000:.1f}ms")
        print(f"Latencia p95: {self.percentil(95)*1000:.1f}ms")
        print(f"Latencia p99: {self.percentil(99)*1000:.1f}ms")

# Test con datos simulados
import random
mc = MetricsCollector()
for _ in range(100):
    latencia = random.gauss(0.1, 0.03)   # ~100ms promedio
    mc.registrar(max(0.01, latencia), random.randint(10, 50))
mc.resumen()

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 1

class MetricsCollector:
    def __init__(self):
        self.latencias = []
        self.tokens_totales = 0
        self.inicio = time.time()
    
    def registrar(self, duracion_segundos: float, tokens_generados: int):
        self.latencias.append(duracion_segundos)
        self.tokens_totales += tokens_generados
    
    def tokens_por_segundo(self) -> float:
        elapsed = time.time() - self.inicio
        return self.tokens_totales / elapsed if elapsed > 0 else 0.0
    
    def percentil(self, p: float) -> float:
        if not self.latencias:
            return 0.0
        ordenadas = sorted(self.latencias)
        idx = int(len(ordenadas) * p / 100)
        idx = min(idx, len(ordenadas) - 1)
        return ordenadas[idx]
    
    def resumen(self):
        print(f"Throughput:    {self.tokens_por_segundo():.1f} tokens/s")
        print(f"Latencia p50:  {self.percentil(50)*1000:.1f}ms")
        print(f"Latencia p95:  {self.percentil(95)*1000:.1f}ms")
        print(f"Latencia p99:  {self.percentil(99)*1000:.1f}ms")

mc = MetricsCollector()
for _ in range(100):
    latencia = random.gauss(0.1, 0.03)
    mc.registrar(max(0.01, latencia), random.randint(10, 50))
mc.resumen()

## ✏️ Ejercicio 2: `PriorityScheduler`

**Problema:** Implementa un scheduler donde los requests VIP tienen prioridad.

In [ ]:
# ✏️ EJERCICIO 2 — Completa el TODO
import heapq

class PriorityScheduler:
    """Scheduler con soporte de prioridades. Prioridad menor = más urgente."""
    
    def __init__(self):
        self._heap = []    # min-heap: (prioridad, id_request, request)
        self._counter = 0  # Para desempate determinista
    
    def submit(self, request: Request, priority: int = 5):
        """Añade una request con prioridad (0 = más urgente, 9 = menos urgente)."""
        # TODO: Insertar en el heap con la prioridad correcta
        pass
    
    def next(self) -> Optional[Request]:
        """Retorna la siguiente request de mayor prioridad. None si está vacío."""
        # TODO: Extraer del heap
        pass
    
    def size(self) -> int:
        return len(self._heap)

# Test
from typing import Optional
sched = PriorityScheduler()
sched.submit(Request("req-normal", "prompt", max_tokens=5), priority=5)
sched.submit(Request("req-vip", "prompt vip", max_tokens=5), priority=1)
sched.submit(Request("req-normal2", "otro prompt", max_tokens=5), priority=5)

print("Orden de despacho:")
while sched.size() > 0:
    r = sched.next()
    print(f"  {r.id}")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 2

class PriorityScheduler:
    def __init__(self):
        self._heap = []
        self._counter = 0
    
    def submit(self, request: Request, priority: int = 5):
        heapq.heappush(self._heap, (priority, self._counter, request))
        self._counter += 1
    
    def next(self) -> Optional[Request]:
        if not self._heap:
            return None
        _, _, request = heapq.heappop(self._heap)
        return request
    
    def size(self) -> int:
        return len(self._heap)

sched = PriorityScheduler()
sched.submit(Request("req-normal", "prompt", max_tokens=5), priority=5)
sched.submit(Request("req-vip", "prompt vip", max_tokens=5), priority=1)
sched.submit(Request("req-normal2", "otro prompt", max_tokens=5), priority=5)

print("Orden de despacho (menor número = mayor prioridad):")
while sched.size() > 0:
    r = sched.next()
    print(f"  {r.id}")

## 🎯 Quiz — Preguntas de Entrevista

**P1:** ¿Por qué el KV Cache solo funciona para los tokens YA generados?
> **R:** Porque el transformer necesita ver todos los tokens anteriores (atención completa).
> Los K,V del prompt se calculan UNA vez y se reutilizan. Los nuevos tokens agregan sus K,V.

**P2:** ¿Qué es el "memory cliff" en inferencia de LLMs?
> **R:** El punto donde el KV Cache no cabe en VRAM. Un modelo con contexto de 128k tokens
> puede necesitar decenas de GB solo para el caché. Soluciones: PagedAttention (vLLM), offloading.

**P3:** ¿Cuál es la diferencia entre latencia TTFT y TPOT?
> **R:** TTFT (Time To First Token): tiempo hasta el primer token → mide la latencia percibida.
> TPOT (Time Per Output Token): tiempo entre tokens sucesivos → mide el throughput.

**P4:** ¿Por qué continuous batching mejora sobre static batching?
> **R:** En static batching, cuando una request termina, el slot de GPU queda vacío hasta que
> TODAS las del batch terminen. Continuous batching reemplaza requests completadas inmediatamente.